# Wie sich die Quellen bewegen

**Zweck.** Für die Auswertung zählt der aktuelle Stand. Aber beide Quellen
ändern ihre Vergangenheit: Detroit korrigiert Einträge noch Jahre nach der Tat,
Chicago liefert Deltas und klassiert um. Diese Bewegung ist selbst ein
Gegenstand — *was ändert sich, wie oft, welcher Art, und was verschwindet.*

`silver/source_change` zieht sie aus aufeinanderfolgenden Vollschnappschüssen
heraus, bevor die Rohläufe entbehrlich werden. Dieses Notebook führt vor, was
dabei herauskommt, wie die Tabelle gebaut ist — und wo ihre Grenzen liegen.

**Was dieses Notebook nicht behauptet.** Es beschreibt zwei Vergleichsfenster:
Detroit vom 11. bis 18.9.2026 (vier Paare in sieben Tagen) und Chicago vom
5.9. bis 3.10.2026 (ein Paar über 28 Tage). Das ist zu wenig für eine Aussage
über Quellbewegung *im Allgemeinen*. Jede Zahl hier gilt für ihr Fenster, und
alle Zahlen sind unten aus der Tabelle gelesen, nicht abgeschrieben.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config").exists():
    raise RuntimeError("Projektwurzel nicht gefunden")
sys.path.insert(0, str(PROJECT_ROOT / "src" / "crime"))

import duckdb

from storage import parquet_spalten

TABELLE = PROJECT_ROOT / "data" / "silver" / "dataset=source_change" / "part-0001.parquet"
if not TABELLE.is_file():
    raise RuntimeError(
        "silver/source_change ist nicht gebaut. "
        "Zuerst: python -m silver_pipeline.build_source_change")

con = duckdb.connect()
QUELLE = f"read_parquet('{TABELLE}')"


def frage(sql):
    """Eine Abfrage auf die Tabelle, als Liste von Zeilen."""
    return con.execute(sql).fetchall()


print(f"DuckDB {duckdb.__version__}")
print(f"Tabelle: {TABELLE.stat().st_size / 1024:.0f} KB")

## Die Spalten — und warum sie ausgeschrieben werden

Eine Zeile je **geändertem Feld**, nicht je geänderter Zeile. Ein Delikt, an
dem sich drei Felder bewegt haben, steht dreimal darin.

Gelesen werden die Spalten über `storage.parquet_spalten` und nicht über
`DESCRIBE`: Der Pfad trägt `dataset=source_change`, und DuckDB leitete daraus
eine Spalte `dataset` ab, die in keiner Datei steht.

In [ ]:
for name in parquet_spalten(TABELLE):
    print(" ", name)

## Wie die Tabelle gebaut ist

Verglichen werden **aufeinanderfolgende Paare von Vollschnappschüssen** — nicht
jedes gegen jedes. Die Kette zählt jede Bewegung genau einmal.

Nur Vollabzüge: Ein Delta enthält bloss die bewegten Zeilen, und *was fehlt*
liesse sich darin nicht von *was sich nicht geändert hat* unterscheiden —
jede Zeile ausserhalb des Deltas sähe verschwunden aus.

Fünf Arten von Beobachtung:

| `art` | Bedeutung |
| --- | --- |
| `geaendert` | etwas ausserhalb von Rauschen und Zeitstempeln hat sich bewegt |
| `beruehrt` | **nur** Zeitstempel haben sich bewegt — die Quelle hat die Zeile angefasst, ohne dass sich fachlich etwas änderte |
| `neu` | der Schlüssel steht im zweiten Schnappschuss und nicht im ersten |
| `verschwunden` | umgekehrt |
| `unveraendert` | eine Markierungszeile für ein Paar, in dem sich nichts bewegt hat |

Die letzte ist die unauffälligste und die wichtigste: Ohne sie wäre *nichts
gefunden* nicht von *noch nicht verglichen* zu unterscheiden, und das Paar
bliebe für immer als unerledigt gesperrt.

**Zwei Entscheidungen, die die Tabelle erst brauchbar machen.** Detroits
`ESRI_OID` wird nie aufgezeichnet — es ist die Zeilennummer der ArcGIS-Antwort
und bewegte sich bei 88'677 von 88'685 Zeilen, ein Faktor 870 Rauschen. Und
jedes Feld wird vor dem Vergleich getrimmt: Von 160 Änderungen an
`offense_description` waren 150 reine Leerzeichen, weil die Quelle inzwischen
auf feste Breite auffüllt.

## Was drin ist

In [ ]:
def zahl(n):
    """Schweizer Tausendertrenner - nur auf der Zahl, nicht dem Satz."""
    return f"{n:,}".replace(",", "'")


zeilen = frage(f"SELECT count(*) FROM {QUELLE}")[0][0]
delikte = frage(f"SELECT count(DISTINCT offense_id) FROM {QUELLE} "
                "WHERE offense_id IS NOT NULL")[0][0]
print(f"{zahl(zeilen)} Zeilen, {zahl(delikte)} betroffene Delikte")

print("\nje Stadt und Art:")
for stadt, art, n in frage(
        f"SELECT city, art, count(*) FROM {QUELLE} GROUP BY 1, 2 ORDER BY 1, 3 DESC"):
    print(f"  {stadt:9} {art:14} {zahl(n):>6}")

In [ ]:
print("Vergleichspaare, älteste zuerst:")
for stadt, von, bis, tage, n in frage(f"""
        SELECT city,
               min(vorher_am)::DATE, max(beobachtet_am)::DATE,
               datediff('day', min(vorher_am), max(beobachtet_am)),
               count(*)
        FROM {QUELLE}
        GROUP BY city, vorher_run_id, run_id
        ORDER BY 2"""):
    wort = "Tag " if tage == 1 else "Tage"
    print(f"  {stadt:9} {von} -> {bis}  {tage:>3} {wort}  {zahl(n):>6} Zeilen")

## Die Gegenprobe, die das Zählwerk prüft

Chicagos Paar lässt sich gegen eine Zahl halten, die **nicht** aus dieser
Tabelle kommt: die Zeilenzahl der beiden Bronze-Manifeste. Wenn `neu` minus
`verschwunden` genau der Differenz entspricht, hat der Vergleich keine Zeile
verloren und keine erfunden.

In [ ]:
import json as _json

laeufe = sorted(
    (PROJECT_ROOT / "data" / "bronze" / "city=chicago").rglob("manifest.json"))
voll = [_json.loads(p.read_text()) for p in laeufe]
voll = sorted((m for m in voll if m.get("mode") == "full_snapshot"),
              key=lambda m: m["started_at_utc"])

if len(voll) < 2:
    print("weniger als zwei Chicago-Vollabzüge - keine Gegenprobe möglich")
else:
    erst, letzt = voll[0], voll[-1]
    differenz = letzt["row_count"] - erst["row_count"]
    neu, weg = frage(f"""
        SELECT count(*) FILTER (WHERE art = 'neu'),
               count(*) FILTER (WHERE art = 'verschwunden')
        FROM {QUELLE} WHERE city = 'chicago'""")[0]
    print(f"Schnappschuss {erst['started_at_utc'][:10]}: {zahl(erst['row_count'])} Zeilen")
    print(f"Schnappschuss {letzt['started_at_utc'][:10]}: {zahl(letzt['row_count'])} Zeilen")
    print(f"Differenz der Manifeste          : {differenz:+d}")
    print(f"neu ({neu}) minus verschwunden ({weg}) : {neu - weg:+d}")
    print("stimmt" if neu - weg == differenz else "WEICHT AB")

## Der Befund: die beiden Städte bewegen sich unterschiedlich

Das ist die Aussage dieses Notebooks. Beide Quellen ändern ihre Vergangenheit,
aber sie ändern **verschiedene Dinge** — und das ist für die Vergleichbarkeit
beider Städte eine Eigenschaft der Quellen, nicht ein Messfehler.

In [ ]:
STEMPEL = {"detroit": ("updated_at", "updated_in_ibr_at", "case_status_updated_at"),
           "chicago": ("updated_on",)}

for stadt, felder in STEMPEL.items():
    liste = ", ".join(f"'{f}'" for f in felder)
    gesamt, stempel = frage(f"""
        SELECT count(*), count(*) FILTER (WHERE feld IN ({liste}))
        FROM {QUELLE} WHERE city = '{stadt}' AND feld IS NOT NULL""")[0]
    if not gesamt:
        continue
    fachlich = gesamt - stempel
    print(f"{stadt}: {gesamt} Feldbewegungen - "
          f"{stempel} Zeitstempel ({stempel/gesamt:.0%}), "
          f"{fachlich} fachlich ({fachlich/gesamt:.0%})")
    for feld, n in frage(f"""
            SELECT feld, count(*) FROM {QUELLE}
            WHERE city = '{stadt}' AND feld IS NOT NULL AND feld NOT IN ({liste})
            GROUP BY 1 ORDER BY 2 DESC LIMIT 6"""):
        print(f"    {feld:26} {n:>5}  ({n/fachlich:.0%} der fachlichen)")
    print()

### Detroit führt den Verfahrensstand nach

Die fachliche Bewegung ist fast vollständig `case_status`. Das passt zur
Quelle: Ein Fall wird erfasst, und was sich später ändert, ist sein Stand im
Verfahren — nicht die Tat.

### Chicago klassiert um und trägt Verhaftungen nach

Hier bewegen sich die Deliktcodes selbst. Ein paar Beispiele, wörtlich aus der
Tabelle:

In [ ]:
for feld, alt, neu, n in frage(f"""
        SELECT feld, wert_alt, wert_neu, count(*) FROM {QUELLE}
        WHERE city = 'chicago' AND feld IS NOT NULL
        GROUP BY 1, 2, 3 ORDER BY 4 DESC, 1 LIMIT 12"""):
    mal = f"  ({n}x)" if n > 1 else ""
    print(f"  {feld:20} {str(alt)[:30]!r:34} -> {str(neu)[:30]!r}{mal}")

`arrest` von `false` auf `true` heisst: Die Verhaftung kam nach der Erfassung.
Sie geht aber auch in die andere Richtung — ein Haftvermerk wird
zurückgenommen. Beide Richtungen getrennt gezählt, und danach die Frage, ob
sich die drei Deliktcodes **gemeinsam** bewegen oder einzeln. Einzeln wäre ein
Widerspruch in der Quelle: `primary_type`, `iucr` und `fbi_code` codieren
dieselbe Tat auf drei Weisen.

In [ ]:
hin, zurueck = frage(f"""
    SELECT count(*) FILTER (WHERE wert_alt = 'false' AND wert_neu = 'true'),
           count(*) FILTER (WHERE wert_alt = 'true' AND wert_neu = 'false')
    FROM {QUELLE} WHERE feld = 'arrest'""")[0]
print(f"arrest: {hin}x false -> true (Verhaftung nachgetragen), "
      f"{zurueck}x true -> false (zurueckgenommen)")

CODES = ("primary_type", "iucr", "fbi_code", "description")
liste = ", ".join(f"'{f}'" for f in CODES)
print("\nChicago-Delikte nach Zahl gleichzeitig bewegter Deliktcodes:")
verteilung = frage(f"""
    SELECT anzahl, count(*) FROM (
        SELECT offense_id, count(DISTINCT feld) AS anzahl FROM {QUELLE}
        WHERE city = 'chicago' AND feld IN ({liste}) GROUP BY 1)
    GROUP BY 1 ORDER BY 1""")
for wieviele, delikte in verteilung:
    wort = "Delikt" if delikte == 1 else "Delikte"
    print(f"  {wieviele} von {len(CODES)}: {delikte} {wort}")
if verteilung:
    einzeln = sum(n for w, n in verteilung if w == 1)
    gruppe = sum(n for w, n in verteilung if w > 1)
    print(f"\n  {gruppe} Delikte bewegten mehrere Codes gemeinsam, "
          f"{einzeln} nur einen einzelnen.")
    print("  -> die Codes beschreiben dieselbe Umklassierung" if gruppe > einzeln
          else "  -> die Codes bewegen sich unabhaengig - das waere ein Widerspruch")

**Eine Bewegung verdient besondere Aufmerksamkeit:** Wenn sich `date` selbst
ändert, verschiebt sich die Tat in der Zeit. Jede Zeitreihe, jede Monatszelle
in Gold und jede Rate je 1'000 Einwohner für diesen Monat ist davon betroffen.

In [ ]:
treffer = frage(f"""
    SELECT city, offense_id, wert_alt, wert_neu FROM {QUELLE}
    WHERE feld IN ('date', 'incident_occurred_at')""")
if treffer:
    for stadt, delikt, alt, neu in treffer:
        print(f"  {stadt:9} {delikt:16} {alt} -> {neu}")
else:
    print("keine Verschiebung des Tatzeitpunkts in den verglichenen Paaren")

## Was verschwindet

`verschwunden` heisst: Der Schlüssel stand im älteren Vollschnappschuss und
steht im neueren nicht mehr. Für eine Abgabe ist das die unangenehmste Art von
Bewegung — eine Zahl, die einmal berichtet wurde, lässt sich nicht mehr
reproduzieren, wenn die Quelle den Fall zurückgezogen hat.

In [ ]:
weg = frage(f"""
    SELECT city, offense_id, vorher_am::DATE, beobachtet_am::DATE
    FROM {QUELLE} WHERE art = 'verschwunden' ORDER BY 1, 2""")
print(f"{len(weg)} verschwundene Delikte")
for stadt, delikt, von, bis in weg:
    print(f"  {stadt:9} {delikt:16} zwischen {von} und {bis}")

## Was die Tabelle kostet — und was sie ersetzt

Das ist das Argument, aus dem sie entstanden ist. `docs/intern/datenhistorie.md`
hat vorgerechnet, was beim Aufräumen verlorengeht: **31,9 MB Schnappschüsse für
102 geänderte Zeilen.** Die Tabelle zieht den Unterschied heraus, bevor die
Läufe entbehrlich werden.

In [ ]:
bronze = sum(p.stat().st_size
             for stadt in ("chicago", "detroit")
             for p in (PROJECT_ROOT / "data" / "bronze" / f"city={stadt}").rglob("*.parquet"))
tabelle = TABELLE.stat().st_size
print(f"Bronze-Läufe beider Städte: {bronze / 1_048_576:>8.1f} MB")
print(f"silver/source_change      : {tabelle / 1_048_576:>8.1f} MB")
print(f"Verhältnis                : {bronze / tabelle:>8.0f} : 1")

## Die Grenzen, ausdrücklich

1. **Nur Vollabzüge.** Zwischen zwei Deltas lässt sich nicht vergleichen. Seit
   dem 3.10.2026 gilt deshalb: ein Vollschnappschuss je Stadt und Monat.
2. **Nur was zwischen zwei Aufnahmen sichtbar ist.** Zwei Änderungen im
   Intervall fallen auf eine zusammen, und eine Änderung samt ihrer Rücknahme
   ist unsichtbar. Das liegt in der Natur des Schnappschussvergleichs.
3. **Die Fenster sind kurz und ungleich** — Detroit sieben Tage, Chicago 28.
   Eine Rate „Änderungen je Monat" liesse sich daraus rechnen und wäre
   irreführend, weil Detroits Fenster vier Mal kürzer ist.
4. **Was nicht aufgezeichnet wird, fehlt hier auch**: Detroits `ESRI_OID` als
   Rauschen, und reine Leerzeichenänderungen durch das Trimmen. Beides mit
   Begründung — siehe den Modulkopf von `build_source_change`.
5. **Chicagos `rauschen` ist leer, und das ist gemessen**: `location`,
   `latitude`, `longitude` und `_extra` bewegten sich im Paar 5.9./3.10. nicht,
   obwohl `location` die Koordinaten doppelt führt.

Mit jedem Monat kommt je Stadt ein Paar hinzu. Die Fragen, die dieses Notebook
heute nicht beantworten kann — wie schnell sich ein Jahrgang stabilisiert, ob
Umklassierungen eine Richtung haben, ob Löschungen ein Muster zeigen — werden
mit dem Takt beantwortbar, nicht mit einer besseren Abfrage.